# 16 · Distillation, and a loss that weights edges

Two ways to get more out of a model without touching its graph - so neither changes what it
costs in the browser:

- **Distillation.** The cheap tier is the luma 32-16 body (17,080 operations an LR pixel, 2.6 ms
  a 1080p frame, against notebook 8's 28,120 and 3.9). Every notebook trains it against the
  truth alone. Here it is also trained against notebook 8's told model, the best model in the
  project: a picture a 32-16 can come closer to than the truth, made by a model that was told
  the host's path, which the student is not.
- **A loss that weights edges.** Every model since notebook 2 trains on `metrics.ssim_l1_loss`,
  whose L1 counts a pixel of flat background like a pixel of a glyph's edge - and on a screen
  the edges are the picture. Two variants: the L1 weighted by how strong an edge the truth has
  at each pixel, and `ssim_l1_loss` plus a term on the luma gradients themselves.

**Smoke tests first, then the two that moved, trained to convergence.** Every smoke arm trains
2 epochs from a trained model against a control that trains the same 2 epochs on the same
batches with the plain loss, so the gap between an arm and its control is the idea and nothing
else; the control at a second seed (another batch order) is the ruler. Then the two ideas with
a direction are trained from the start until they converge, each beside the plain recipe at
two seeds - the ruler again - and the same seed's batches.

Steps:

1. **Data** - notebook 8's patch set, every batch with what its patches were told.
2. **Models and losses** - the teacher, the student, the two edge losses.
3. **Distillation** - the student fine-tuned 2 epochs, with and without the teacher.
4. **Edge losses** - notebook 8's model fine-tuned 2 epochs with each loss.
5. **Trained to convergence** - the 32-16 plain and distilled, then the same on a leaky ReLU
   that cannot switch a channel off; notebook 8's told model with the edge-weighted loss, at
   both seeds.
6. **What it says.**
7. **Export** - the cheap tier, the luma 32-16 leaky, at the client's tile.

Needs: notebook 8's `checkpoints/upscale_screens.pt` and its runs in `checkpoints/screens_runs/`,
and notebook 4's run `checkpoints/strategy_runs/luma_32-16_seed0_lr0.001.pt`. Writes the cheap tier's
graphs to `dist/upscale/upscale_luma_32-16_leaky_*.onnx` and each full training to
`checkpoints/distill_runs/`, saved every epoch: run the cell again after an
interruption and it carries on; finished runs just load. The smoke tests keep nothing.

In [1]:
"""Settings."""

import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from helpers import degrade, metrics, models, training, webexport
from helpers.training import DATA_DIR, DEVICE

TEACHER = training.CHECKPOINT_DIR / "upscale_screens.pt"                                  # notebook 8's told model
STUDENT_START = training.CHECKPOINT_DIR / "strategy_runs" / "luma_32-16_seed0_lr0.001.pt"   # notebook 4's luma 32-16
STUDENT = dict(widths=(32, 16), blocks=0, stem=5)                                          # its shape

SMOKE_EPOCHS = 2          # every arm, and its control
BATCH_SIZE = 32
SEEDS = (0, 1)            # the control at both: how far the batch order alone moves 2 epochs
STUDENT_RATE = 1e-3       # the recipe's: the student is learning a set it has not seen
FINE_TUNE_RATE = 1e-4     # notebook 8's model has converged; 1e-3 knocks it off its minimum
DISTILL_WEIGHTS = (0.5, 1.0)   # the share of the student's loss that is against the teacher
EDGE_STRENGTH = 4.0       # an average edge pixel weighs 1 + this, a flat one 1 (before scaling to mean 1)
GRADIENT_WEIGHT = 1.0     # the gradient term, on top of ssim_l1_loss

RUN_DIR = training.CHECKPOINT_DIR / "distill_runs"       # each full training, saved every epoch
TOLD_RUNS = {seed: training.CHECKPOINT_DIR / "screens_runs" / f"told_the_bitrate_and_path_seed{seed}.pt"
             for seed in SEEDS}                           # notebook 8's told model, at both seeds
MAX_EPOCHS = 500          # a safety limit; each training stops once it has converged
LEARNING_RATE = 1e-3      # notebook 8's recipe, for every full training
FULL_DISTILL_WEIGHT = 0.5 # the share that keeps the SSIM term: the recipe selects on SSIM-Y
LEAKY_SLOPE = 0.1         # notebook 4's `32-16 leaky`: a negative value passes at a tenth

training.seed_everything(SEEDS[0])

## 1. Data

The patch set of `00_data_preprocess.ipynb`, with screens, as notebook 8 trained on. Every batch
carries what its patches were told (the bitrate and the host's path, `degrade.condition`): the
teacher reads it, every other model drops it. Scores are split by what a patch shows.

In [2]:
manifest = training.load_manifest()
training.describe(manifest)
SCALE = manifest["scale"]
trim = training.trimmer(manifest)
teacher_file = training.load_checkpoint(TEACHER)
TOLD = tuple(teacher_file["told"])


def told_condition(source):
    return degrade.condition(source["degradation"], TOLD)


train_loader, val_loader = training.loaders(DATA_DIR, manifest, BATCH_SIZE, val_batch_size=128,
                                            condition=told_condition)


def content(source):
    """What a source shows: a drawn desktop screen, a photo (as a moving clip) or a video."""
    if source["dataset"] == "screens":
        return "screen"
    return "photo" if "motion" in source else "video"


groups = {}   # content -> the indices of its validation patches
for source in manifest["splits"]["val"]["sources"]:
    first, end = source["patches"]
    groups.setdefault(content(source), []).extend(range(first, end))
group_loaders = {name: DataLoader(Subset(val_loader.dataset, indices), batch_size=128)
                 for name, indices in sorted(groups.items())}
print("val by content: " + "  ".join(f"{name} {len(indices):,}" for name, indices in sorted(groups.items())))

device  cuda NVIDIA GeForce RTX 3090
scale   2
patch   72 px LR in (64 px + a halo of 4) -> 128 px HR out, 8 frames each
train  17140 patches from 1089 sources
val     3523 patches from 225 sources
val by content: photo 784  screen 1,154  video 1,585


## 2. Models and losses

**The teacher** is notebook 8's told model as it was saved (`models.ToldLuma`), frozen. **The
student** is notebook 4's luma 32-16 at its best epoch: trained to convergence on the set before
it had screens, so 2 epochs here are mostly it learning screens - with or without help. It is not
told, so `Blind` drops the condition.

The two losses, both on luma gradients (a Sobel, horizontal and vertical):

- `edge_weighted_loss` - `ssim_l1_loss` with its L1 weighted per pixel: 1 on a flat background,
  1 + `EDGE_STRENGTH` at an edge as strong as the set's average. The weights are scaled to
  average 1 over a batch, so the L1 keeps its share against the SSIM term; what moves is which
  pixels it is spent on.
- `gradient_loss` - `ssim_l1_loss` plus the L1 between the prediction's and the truth's luma
  gradients: an edge that is soft, shifted or doubled costs again.

In [3]:
class Blind(nn.Module):
    """A model that is not told, run on loaders that tell: it drops the condition."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, condition=None):
        return self.model(x)


def teacher_model():
    """Notebook 8's told model, as it was saved."""
    model = models.ToldLuma(scale=teacher_file["scale"], **teacher_file["arguments"]).to(DEVICE)
    model.load_state_dict(teacher_file["state_dict"])
    return model


def student_model():
    """Notebook 4's luma 32-16 at its best epoch, trained before the set had screens."""
    model = models.LumaUpscaler(**STUDENT, scale=SCALE).to(DEVICE)
    model.load_state_dict(torch.load(STUDENT_START, map_location=DEVICE, weights_only=True)["best"]["state"])
    return Blind(model)


teacher = teacher_model().eval()
assert max(teacher.halo, student_model().model.halo) <= manifest["halo"], "a model reaches further than the halo"
print(f"teacher: {models.count_parameters(teacher, True):,} learned parameters, "
      f"student: {models.count_parameters(student_model(), True):,}")
teacher.requires_grad_(False)

SOBEL = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]) / 8


def luma(x):
    return metrics.rgb_to_yuv(x)[:, :1]


def gradients(y):
    """Horizontal and vertical Sobel of (N, 1, H, W) luma, as (N, 2, H - 2, W - 2)."""
    kernel = torch.stack([SOBEL, SOBEL.T])[:, None].to(y.device, y.dtype)
    return F.conv2d(y, kernel)


def edge_strength(y):
    """How strong an edge (N, 1, H, W) luma has at each pixel, the same size (edge repeated)."""
    return F.pad(gradients(y).norm(dim=1, keepdim=True), (1, 1, 1, 1), mode="replicate")


# the set's average edge strength in the truth, which EDGE_STRENGTH is counted against
with torch.no_grad():
    torch.manual_seed(SEEDS[0])
    EDGE_SCALE = float(np.mean([edge_strength(luma(hr.to(DEVICE))).mean().item()
                                for _, (_, hr) in zip(range(20), train_loader)]))
print(f"average luma edge strength in the truth: {EDGE_SCALE:.4f}")


def edge_weighted_loss(prediction, target, alpha=0.84, strength=EDGE_STRENGTH):
    """`ssim_l1_loss` with its L1 weighted by how strong an edge the truth has at each pixel."""
    with torch.no_grad():
        weight = 1 + strength * edge_strength(luma(target)) / EDGE_SCALE
        weight = weight / weight.mean()
    structure = 1 - metrics._ssim(luma(prediction), luma(target)).mean()
    return alpha * structure + (1 - alpha) * (weight * (prediction - target).abs()).mean()


def gradient_term(prediction, target):
    return (gradients(luma(prediction)) - gradients(luma(target))).abs().mean()


def gradient_loss(prediction, target, weight=GRADIENT_WEIGHT):
    """`ssim_l1_loss` plus the L1 between the luma gradients of the prediction and the truth."""
    return metrics.ssim_l1_loss(prediction, target) + weight * gradient_term(prediction, target)


# what each term is worth at the start, on the teacher's own output: the share a weight buys
with torch.no_grad():
    (lr, condition), hr = next(iter(val_loader))
    lr, condition, hr = lr.to(DEVICE), condition.to(DEVICE), hr.to(DEVICE)
    out = trim(teacher(lr, condition))
    print(f"on the teacher's output: ssim_l1_loss {metrics.ssim_l1_loss(out, hr):.4f}, "
          f"edge_weighted_loss {edge_weighted_loss(out, hr):.4f}, gradient term {gradient_term(out, hr):.4f} "
          f"(x {GRADIENT_WEIGHT:g})")

teacher: 13,588 learned parameters, student: 7,636
average luma edge strength in the truth: 0.0292
on the teacher's output: ssim_l1_loss 0.1395, edge_weighted_loss 0.1426, gradient term 0.0124 (x 1)


In [4]:
def fine_tune(model, loss_fn, rate, seed, teacher_weight=0.0):
    """Train `model` `SMOKE_EPOCHS` epochs from where it is. With a `teacher_weight`, that share
    of the loss is L1 against the teacher's picture and the rest `loss_fn` against the truth.
    The same seed draws the same batches, flips and frames for every arm. Returns the model."""
    torch.manual_seed(seed)
    optimiser = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=rate)
    started = time.time()
    for epoch in range(SMOKE_EPOCHS):
        model.train()
        for (lr, condition), hr in train_loader:
            lr, condition, hr = lr.to(DEVICE), condition.to(DEVICE), hr.to(DEVICE)
            prediction = trim(model(lr, condition))
            loss = loss_fn(prediction, hr)
            if teacher_weight:
                with torch.no_grad():
                    taught = trim(teacher(lr, condition))
                loss = (1 - teacher_weight) * loss + teacher_weight * F.l1_loss(prediction, taught)
            optimiser.zero_grad(set_to_none=True)
            loss.backward()
            optimiser.step()
    print(f"  {SMOKE_EPOCHS} epochs in {training.format_time(time.time() - started)}")
    return model.eval()


def scores(model):
    """`training.evaluate` on the whole validation set and on each content."""
    return {"all": training.evaluate(model, val_loader, trim),
            **{name: training.evaluate(model, loader, trim) for name, loader in group_loaders.items()}}


def distill_loss(prediction, target, taught, weight=FULL_DISTILL_WEIGHT):
    """`weight` of the loss L1 against the teacher's picture, the rest `ssim_l1_loss` against
    the truth (the convention of `training.train`'s `teacher`)."""
    return (1 - weight) * metrics.ssim_l1_loss(prediction, target) + weight * F.l1_loss(prediction, taught)


def print_against(results, control_names):
    """SSIM-Y and luma PSNR of every arm, overall and by content, as a gap to the mean of the
    controls; the controls' spread is the ruler. The word is the SSIM-Y verdict: luck within
    the ruler, + or - outside it."""
    columns = ["all", *group_loaders]
    control = {c: {k: np.mean([results[n][c][k] for n in control_names]) for k in ("ssim", "y")} for c in columns}
    ruler = {c: {k: max(results[n][c][k] for n in control_names) - min(results[n][c][k] for n in control_names)
                 for k in ("ssim", "y")} for c in columns}
    width = max(len(name) for name in results) + 2
    print(f"{'':<{width}}{'ssim-y':>9}{'psnr-y':>8}" + "".join(f"{c + ', ssim-y / dB':>26}" for c in columns))
    print(f"{'ruler: the controls apart':<{width}}{'':17}"
          + "".join(f"{ruler[c]['ssim']:.4f} / {ruler[c]['y']:.2f}     ".rjust(26) for c in columns))
    for name, by in results.items():
        cells = ""
        for c in columns:
            gap_ssim, gap_y = by[c]["ssim"] - control[c]["ssim"], by[c]["y"] - control[c]["y"]
            word = "" if name in control_names else (" luck" if abs(gap_ssim) <= ruler[c]["ssim"] else " +" if gap_ssim > 0 else " -")
            cells += f"{gap_ssim:+.4f} / {gap_y:+.2f}{word:<5}".rjust(26)
        print(f"{name:<{width}}{by['all']['ssim']:9.4f}{by['all']['y']:8.2f}{cells}")

## 3. Distillation

The student fine-tuned 2 epochs at the recipe's rate: the controls on `ssim_l1_loss` against the
truth, at two seeds; the arms at seed 0, with half and then all of the loss against the
teacher. The teacher's own score is printed for the ceiling.

In [5]:
distill = {"teacher (notebook 8, told)": scores(teacher), "student, untrained here": scores(student_model())}
for seed in SEEDS:
    name = f"control, seed {seed}"
    print(name)
    distill[name] = scores(fine_tune(student_model(), metrics.ssim_l1_loss, STUDENT_RATE, seed))
for weight in DISTILL_WEIGHTS:
    name = f"distilled, {weight:.0%} teacher"
    print(name)
    distill[name] = scores(fine_tune(student_model(), metrics.ssim_l1_loss, STUDENT_RATE, SEEDS[0], weight))
print()
print_against(distill, [f"control, seed {seed}" for seed in SEEDS])

control, seed 0
  2 epochs in 8s
control, seed 1
  2 epochs in 8s
distilled, 50% teacher
  2 epochs in 9s
distilled, 100% teacher
  2 epochs in 9s

                               ssim-y  psnr-y          all, ssim-y / dB        photo, ssim-y / dB       screen, ssim-y / dB        video, ssim-y / dB
ruler: the controls apart                            0.0013 / 0.02             0.0010 / 0.04             0.0041 / 0.21             0.0005 / 0.16     
teacher (notebook 8, told)     0.9369   34.94      +0.0106 / +0.88 +         +0.0076 / +0.34 +         +0.0211 / +1.37 +         +0.0044 / +0.80 +   
student, untrained here        0.9249   34.20      -0.0014 / +0.14 -         +0.0025 / +0.22 +         -0.0073 / -0.34 -         +0.0011 / +0.45 +   
control, seed 0                0.9270   34.05      +0.0007 / -0.01           +0.0005 / -0.02           +0.0021 / +0.10           -0.0003 / -0.08     
control, seed 1                0.9256   34.06      -0.0007 / +0.01           -0.0005 / +0.02          

## 4. Edge losses

Notebook 8's told model fine-tuned 2 epochs at a tenth of the recipe's rate, from its saved
weights: the controls on `ssim_l1_loss`, at two seeds, then each edge loss at seed 0. A loss
that only re-ranks what the model already does should show here within 2 epochs; one that asks
for something the body cannot do will not.

In [6]:
edges = {"notebook 8, as saved": scores(teacher)}
for seed in SEEDS:
    name = f"control, seed {seed}"
    print(name)
    edges[name] = scores(fine_tune(teacher_model(), metrics.ssim_l1_loss, FINE_TUNE_RATE, seed))
for name, loss_fn in (("edge-weighted L1", edge_weighted_loss), ("+ gradient term", gradient_loss)):
    print(name)
    edges[name] = scores(fine_tune(teacher_model(), loss_fn, FINE_TUNE_RATE, SEEDS[0]))
print()
print_against(edges, [f"control, seed {seed}" for seed in SEEDS])

control, seed 0
  2 epochs in 10s
control, seed 1
  2 epochs in 9s
edge-weighted L1
  2 epochs in 10s
+ gradient term
  2 epochs in 10s

                         ssim-y  psnr-y          all, ssim-y / dB        photo, ssim-y / dB       screen, ssim-y / dB        video, ssim-y / dB
ruler: the controls apart                         0.0001 / 0.03             0.0003 / 0.01             0.0003 / 0.04             0.0001 / 0.04     
notebook 8, as saved     0.9369   34.94      +0.0003 / +0.05 +         +0.0004 / -0.00 +         +0.0004 / +0.12 +         +0.0001 / +0.02 luck
control, seed 0          0.9367   34.88      +0.0001 / -0.02           +0.0002 / -0.00           +0.0002 / -0.02           -0.0001 / -0.02     
control, seed 1          0.9366   34.91      -0.0001 / +0.02           -0.0002 / +0.00           -0.0002 / +0.02           +0.0001 / +0.02     
edge-weighted L1         0.9367   34.91      +0.0000 / +0.02 luck      +0.0001 / +0.02 luck      +0.0001 / +0.14 luck      -0.0001 / -0.07 l

## 5. Trained to convergence

Notebook 8's recipe for every run - Adam from 1e-3, batch 32, the epoch with the best SSIM-Y
kept, until it converges - and, as there, the seed sets both the starting weights and the
batches, so two runs at one seed differ only in their loss.

- **The cheap tier**: the luma 32-16 from the start on this set, plain at both seeds, and at
  seed 0 with half its loss against the teacher (`distill_loss`).
- **The edge loss**: notebook 8's told model from the start with `edge_weighted_loss`, at both
  seeds. Its plain twins are notebook 8's own two told runs, loaded from `screens_runs/`.

In [7]:
def full_run(name, build, seed, loss_fn, teacher=None):
    """A fresh `build()` trained to convergence, or its saved training loaded. As in notebook 8,
    the seed draws the starting weights and then, again, the batches. Returns it at its best epoch."""
    torch.manual_seed(seed)
    model = build()
    torch.manual_seed(seed)
    history, best = training.train(model, train_loader, val_loader, trim, MAX_EPOCHS, learning_rate=LEARNING_RATE,
                                   loss_fn=loss_fn, select="ssim", checkpoint=RUN_DIR / f"{name}.pt", verbose=0,
                                   teacher=teacher)
    print(f"{name:<26} best epoch {best['epoch']:3} of {len(history):3}, ssim-y {best['score']:.4f}")
    return model.eval()


def plain_student():
    return Blind(models.LumaUpscaler(**STUDENT, scale=SCALE).to(DEVICE))


def plain_told():
    return models.ToldLuma(scale=SCALE, **teacher_file["arguments"]).to(DEVICE)


def from_run(path, model):
    """A saved training's best epoch, in `model`."""
    model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True)["best"]["state"])
    return model.eval()


cheap = {}
for seed in SEEDS:
    cheap[f"32-16, seed {seed}"] = scores(full_run(f"32-16_seed{seed}", plain_student, seed, metrics.ssim_l1_loss))
cheap["32-16 distilled, seed 0"] = scores(full_run("32-16_distilled_seed0", plain_student, SEEDS[0], distill_loss, teacher))
cheap["the teacher"] = scores(teacher)
print()
print_against(cheap, [f"32-16, seed {seed}" for seed in SEEDS])

32-16_seed0                best epoch 126 of 132, ssim-y 0.9270
32-16_seed1                best epoch 105 of 111, ssim-y 0.9305
32-16_distilled_seed0      best epoch 116 of 128, ssim-y 0.9239

                            ssim-y  psnr-y          all, ssim-y / dB        photo, ssim-y / dB       screen, ssim-y / dB        video, ssim-y / dB
ruler: the controls apart                         0.0035 / 0.23             0.0017 / 0.05             0.0076 / 0.51             0.0015 / 0.12     
32-16, seed 0               0.9270   34.03      -0.0018 / -0.12           -0.0009 / -0.02           -0.0038 / -0.26           -0.0007 / -0.06     
32-16, seed 1               0.9305   34.27      +0.0018 / +0.12           +0.0009 / +0.02           +0.0038 / +0.26           +0.0007 / +0.06     
32-16 distilled, seed 0     0.9239   33.94      -0.0048 / -0.21 -         -0.0014 / +0.01 luck      -0.0109 / -0.53 -         -0.0021 / -0.09 -   
the teacher                 0.9369   34.94      +0.0081 / +0.79 +       

In [8]:
edge = {f"notebook 8 told, seed {seed}": scores(from_run(path, plain_told())) for seed, path in TOLD_RUNS.items()}
for seed in SEEDS:
    edge[f"told, edge-weighted, seed {seed}"] = scores(full_run(f"told_edge_weighted_seed{seed}", plain_told, seed,
                                                                edge_weighted_loss))
print()
print_against(edge, [f"notebook 8 told, seed {seed}" for seed in SEEDS])

told_edge_weighted_seed0   best epoch 116 of 128, ssim-y 0.9374
told_edge_weighted_seed1   best epoch  87 of  93, ssim-y 0.9364

                                ssim-y  psnr-y          all, ssim-y / dB        photo, ssim-y / dB       screen, ssim-y / dB        video, ssim-y / dB
ruler: the controls apart                             0.0002 / 0.02             0.0003 / 0.00             0.0001 / 0.02             0.0001 / 0.06     
notebook 8 told, seed 0         0.9369   34.94      -0.0001 / +0.01           -0.0002 / +0.00           -0.0001 / -0.01           -0.0001 / +0.03     
notebook 8 told, seed 1         0.9371   34.92      +0.0001 / -0.01           +0.0002 / -0.00           +0.0001 / +0.01           +0.0001 / -0.03     
told, edge-weighted, seed 0     0.9374   34.97      +0.0005 / +0.04 +         +0.0002 / +0.01 luck      +0.0008 / +0.16 +         +0.0003 / -0.04 +   
told, edge-weighted, seed 1     0.9364   34.92      -0.0006 / -0.01 -         -0.0007 / -0.00 -         -0.0008 / +0

### Why the two plain 32-16s are 0.0035 apart

Notebook 4 found that at 1e-3 the 32-16 switches off channels of its narrow layer for good - a
ReLU passes no gradient to a channel that is negative everywhere - and, on the patch set before
it had screens, that it cost nothing measurable. Counted here on every validation patch:

In [9]:
@torch.no_grad()
def dead_channels(model):
    """Channels after each ReLU of a plain body that are zero on every validation pixel."""
    peaks = None
    for (lr, _), _ in val_loader:
        h, batch = lr.to(DEVICE), []
        for layer in model.body:
            h = layer(h)
            if isinstance(layer, nn.ReLU):
                batch.append(h.amax(dim=(0, 2, 3)))
        peaks = batch if peaks is None else [torch.maximum(a, b) for a, b in zip(peaks, batch)]
    return [int((peak <= 0).sum()) for peak in peaks]


for name, run in (("32-16, seed 0", "32-16_seed0"), ("32-16, seed 1", "32-16_seed1"),
                  ("32-16 distilled, seed 0", "32-16_distilled_seed0")):
    dead = dead_channels(from_run(RUN_DIR / f"{run}.pt", plain_student()).model)
    print(f"{name:<26} ssim-y {cheap[name]['all']['ssim']:.4f}, dead channels: "
          + ", ".join(f"{d} of {w}" for d, w in zip(dead, STUDENT["widths"])))

32-16, seed 0              ssim-y 0.9270, dead channels: 0 of 32, 8 of 16
32-16, seed 1              ssim-y 0.9305, dead channels: 0 of 32, 2 of 16
32-16 distilled, seed 0    ssim-y 0.9239, dead channels: 0 of 32, 10 of 16


### The cheap tier on a body that cannot die

So the comparison is made again on notebook 4's `32-16 leaky`, as a luma body: every ReLU of the
body a `LeakyReLU` of slope 0.1 (`models.LumaUpscaler(slope=0.1)`), which still passes a
gradient to a channel that is negative everywhere. It is inside the op budget (`LeakyRelu`), and its published graph runs in the
browser at the 32-16's speed (2.6-2.7 ms a 1080p frame, "The browser benchmark"). Plain at both
seeds, distilled at seed 0, with the same loss and recipe as above.

In [10]:
def leaky_student():
    return Blind(models.LumaUpscaler(**STUDENT, scale=SCALE, slope=LEAKY_SLOPE).to(DEVICE))


leaky = {}
for seed in SEEDS:
    leaky[f"32-16 leaky, seed {seed}"] = scores(full_run(f"32-16_leaky_seed{seed}", leaky_student, seed,
                                                         metrics.ssim_l1_loss))
leaky["32-16 leaky distilled, seed 0"] = scores(full_run("32-16_leaky_distilled_seed0", leaky_student, SEEDS[0],
                                                         distill_loss, teacher))
leaky["32-16 (ReLU), seed 1"] = cheap["32-16, seed 1"]
leaky["the teacher"] = cheap["the teacher"]
print()
print_against(leaky, [f"32-16 leaky, seed {seed}" for seed in SEEDS])

32-16_leaky_seed0          best epoch 108 of 126, ssim-y 0.9309
32-16_leaky_seed1          best epoch 138 of 150, ssim-y 0.9312
32-16_leaky_distilled_seed0 best epoch 113 of 125, ssim-y 0.9312

                                  ssim-y  psnr-y          all, ssim-y / dB        photo, ssim-y / dB       screen, ssim-y / dB        video, ssim-y / dB
ruler: the controls apart                               0.0003 / 0.02             0.0003 / 0.01             0.0009 / 0.04             0.0002 / 0.02     
32-16 leaky, seed 0               0.9309   34.36      -0.0001 / +0.01           -0.0002 / +0.01           -0.0005 / +0.02           +0.0001 / +0.01     
32-16 leaky, seed 1               0.9312   34.34      +0.0001 / -0.01           +0.0002 / -0.01           +0.0005 / -0.02           -0.0001 / -0.01     
32-16 leaky distilled, seed 0     0.9312   34.43      +0.0001 / +0.08 luck      +0.0001 / +0.05 luck      -0.0000 / +0.08 luck      +0.0003 / +0.10 +   
32-16 (ReLU), seed 1              0.9305 

### A leaky ReLU on notebook 8's model

Notebook 8's 24 + 2 blocks never showed a dying problem - its residual blocks add their input
back, which keeps a channel alive - but a leaky ReLU is free in the graph, so it is tried there
too: the told model with `slope=0.1`, trained exactly as notebook 8 trained it, at both seeds,
against notebook 8's two told runs.

In [11]:
def leaky_told():
    return models.ToldLuma(scale=SCALE, slope=LEAKY_SLOPE, **teacher_file["arguments"]).to(DEVICE)


told_leaky = {name: edge[name] for name in (f"notebook 8 told, seed {seed}" for seed in SEEDS)}
for seed in SEEDS:
    told_leaky[f"told leaky, seed {seed}"] = scores(full_run(f"told_leaky_seed{seed}", leaky_told, seed,
                                                             metrics.ssim_l1_loss))
print()
print_against(told_leaky, [f"notebook 8 told, seed {seed}" for seed in SEEDS])

told_leaky_seed0           best epoch  99 of 105, ssim-y 0.9367
told_leaky_seed1           best epoch 121 of 139, ssim-y 0.9375

                            ssim-y  psnr-y          all, ssim-y / dB        photo, ssim-y / dB       screen, ssim-y / dB        video, ssim-y / dB
ruler: the controls apart                         0.0002 / 0.02             0.0003 / 0.00             0.0001 / 0.02             0.0001 / 0.06     
notebook 8 told, seed 0     0.9369   34.94      -0.0001 / +0.01           -0.0002 / +0.00           -0.0001 / -0.01           -0.0001 / +0.03     
notebook 8 told, seed 1     0.9371   34.92      +0.0001 / -0.01           +0.0002 / -0.00           +0.0001 / +0.01           +0.0001 / -0.03     
told leaky, seed 0          0.9367   34.94      -0.0003 / +0.01 -         +0.0001 / +0.03 luck      -0.0008 / -0.03 -         -0.0001 / +0.02 luck
told leaky, seed 1          0.9375   35.06      +0.0006 / +0.13 +         +0.0009 / +0.04 +         +0.0006 / +0.17 +         +0.0004 / 

## 6. What it says

**A leaky ReLU is what the cheap tier was missing.** On a ReLU the luma 32-16 is a lottery:
two seeds land 0.0035 SSIM-Y apart, because the narrow layer switches off 8 of its 16 channels
at seed 0 and 2 at seed 1. A leaky ReLU (slope 0.1) cannot switch one off, and the same two
seeds land 0.0003 apart, at 0.9309 and 0.9312 - above the better ReLU seed (0.9305) on every
content but screens, where it is level, and 0.0023 above the ReLU pair's mean. On this set,
with screens, dying costs; notebook 4 measured it free before there were screens. In the browser
the leaky graph (section 7) runs at the ReLU graph's speed - 0.430 ms a float32 tile, 0.356 a
float16 one, **2.75 ms a 1080p frame** as one float16 graph, against 0.430, 0.354 and 2.73 for
notebook 4's luma 32-16 in the same rounds - so the cheap tier is 0.931 SSIM-Y for 2.8 ms: above
notebook 4's model (0.9292, 3.9 ms) and DirNet D (0.9303, 7.7 ms), and far above notebook 11's
`S` table (0.9238, 2.3 ms).

**Distillation buys luma dB, not SSIM-Y.** On the leaky body, with half its loss against the
teacher, the student scores the plain seeds' SSIM-Y (0.9312, luck) and +0.08 dB luma overall
against a ruler of 0.02 - +0.05 on photos, +0.08 on screens, +0.10 on video, each outside its
own ruler. It is one seed, and the recipe selects on SSIM-Y, which it does not move: worth
keeping if luma PSNR is what the cheap tier is chosen on, not otherwise. On the ReLU body it
looked worse (0.9239), but that run lost 10 of its 16 narrow channels and says nothing about
the teacher.

**The edge-weighted loss trades video for screens in dB, and nothing in SSIM-Y.** Seed 0 was
+0.0005 SSIM-Y over notebook 8's two told seeds; seed 1 is -0.0006. The two edge seeds are
0.0010 apart against the plain pair's 0.0002, and their mean (0.9369) is the plain pair's. What
both seeds keep is a direction in dB: screens +0.16 and +0.10 dB luma against a ruler of 0.02,
video -0.04 and -0.09 against 0.06. Weighting the L1 by edges moves where the error goes, and
makes the training less steady; it does not make a better model. Notebook 8's loss stays.

**A leaky ReLU does nothing for notebook 8's model.** Its residual blocks add their input back,
so its channels never died; leaky, its two seeds land at 0.9367 and 0.9375 - a mean of 0.9371
against notebook 8's 0.9370, four times as far apart. Seed 1 is the best single model in the
project (+0.13 dB luma, ahead on every content), but picking it would be picking a seed, not a body.
Notebook 8's graph stays.

**The smoke tests were half right.** They found the edge-weighted loss's direction (screens up,
video down, in dB) and nothing in the gradient term, and they showed distillation moving the
student in dB, which the leaky body kept and the ReLU body hid. Two epochs from a trained model
answer which way a loss pushes, not where it ends.

**Next:**

1. The cheap tier is the luma 32-16 leaky (section 7). On notebook 15's whole streamed clips
   it adds +0.0402 SSIM-Y on screens at the same bits, against notebook 8's +0.0436 and
   notebook 4's +0.0292; the iGPU measurement decides whether a client needs it.
2. Distillation only if the cheap tier is ever chosen on luma dB, and then a second distilled
   seed first.

## 7. Export for the browser

The cheap tier: the luma 32-16 leaky at seed 1, the better of its two seeds, trained on the plain
recipe - distillation is one seed and moves only dB. At the client's tile (a 320x180 step),
float32 and float16, and float16 as a whole 1080p frame, as notebook 8 exports its model.

In [12]:
STEP_WIDTH, STEP_HEIGHT = 320, 180       # LR pixels one client tile adds to the picture (notebook 5)
FRAME_WIDTH, FRAME_HEIGHT = 1920, 1080   # the whole-frame graph
NAME, LABEL = "luma_32-16_leaky", "luma 32-16 leaky"


def check_and_time(info, module, shape, dtype, tolerance):
    """Check a written graph against the torch module, time it on the CPU, print one line."""
    webexport.check_graph(info["path"], module, np.random.rand(1, 3, *shape).astype(np.float32), dtype, tolerance)
    ms = webexport.time_graph(info["path"], shape, dtype)
    webexport.annotate(info, cpu_ms=round(ms, 2))
    print(webexport.summary_line(info, ms))


cheap_model = from_run(RUN_DIR / "32-16_leaky_seed1.pt", leaky_student()).model.cpu()
halo = cheap_model.halo
tile_name = f"tile{STEP_WIDTH}x{STEP_HEIGHT}"
tile_shape = (STEP_HEIGHT + 2 * halo, STEP_WIDTH + 2 * halo)
frame_shape = (FRAME_HEIGHT // SCALE + 2 * halo, FRAME_WIDTH // SCALE + 2 * halo)

fp32 = webexport.export(cheap_model, f"upscale_{NAME}_{tile_name}.onnx", tile_shape, label=f"{LABEL} fp32", halo=halo)
assert not fp32["outside_budget"], fp32["outside_budget"]
check_and_time(fp32, cheap_model, tile_shape, np.float32, tolerance=2e-3)
fp16 = webexport.to_float16(fp32, f"upscale_{NAME}_fp16_{tile_name}.onnx")
webexport.write_metadata(fp16["path"], {"label": f"{LABEL} fp16"})
check_and_time(fp16, cheap_model, tile_shape, np.float16, tolerance=4e-3)
frame = webexport.to_frame(fp16, f"upscale_{NAME}_fp16_{tile_name}_frame.onnx", FRAME_WIDTH, FRAME_HEIGHT)
check_and_time(frame, cheap_model, frame_shape, np.float16, tolerance=4e-3)
work = webexport.count_work(fp32["path"])
print(f"{work['operations'] / work['input_pixels']:,.0f} operations and "
      f"{work['activation_bytes'] / work['input_pixels']:,.0f} bytes per LR pixel")

upscale_luma_32-16_leaky_tile320x180.onnx              35.0 KB    3.03 ms   Addx1, Concatx1, Convx4, DepthToSpacex1, LeakyRelux2
upscale_luma_32-16_leaky_fp16_tile320x180.onnx         18.7 KB    7.98 ms   Addx1, Concatx1, Convx4, DepthToSpacex1, LeakyRelux2
upscale_luma_32-16_leaky_fp16_tile320x180_frame.onnx   18.7 KB   90.23 ms   Addx1, Concatx1, Convx4, DepthToSpacex1, LeakyRelux2
17,080 operations and 1,192 bytes per LR pixel
